In [1]:

#This is not finished yet, I have a lot I would like to explore with the SRILM but it is what I have done so far :) 


In [2]:
#Function taken from Assignment 1

import subprocess
import sys
import os

def run_srilm(command, warnings=False):
    """Runs the provided SRILM command and prints any resulting messages
    
    Arguments
    ---------
    command: str; the SRILM command, as it would be entered in the Terminal
    warnings: bool; whether or not to print warnings alongside output
    """
    if sys.platform == 'win32':
        # This is just so that it works on my Windows machine.
        # For the purposes of this class, just ignore this line!
        command = [r"C:\cygwin64\bin\bash.exe ", "--login", "-c",
            "source ~/.bashrc && cd / && /srilm/bin/cygwin64/" + command]

    stdout = subprocess.PIPE
    if warnings:
        stderr = subprocess.STDOUT
    else:
        stderr = subprocess.PIPE
        
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=stdout, stderr=stderr)
        output_msg = result.stdout
        if output_msg:
            print(output_msg.decode('UTF-8'))
        else:
            print("The command ran successfully")
            
    except subprocess.CalledProcessError as e:
        print("The command did not run, and produced the following error message:\n{}".format(e.stderr.decode('UTF-8')))

In [3]:
#Import the SRILM-ready data set made in the preprocessing nb
from pathlib import Path

file_path = Path("../output/countries/")
txt_files = sorted(file_path.glob("*.txt"))

print(f"Imported {len(txt_files)} files:")
print([f.name for f in txt_files])

Imported 14 files:
['CZ.txt', 'DE.txt', 'EC.txt', 'ECB.txt', 'ECR.txt', 'EP.txt', 'EUCouncil.txt', 'GR.txt', 'IMF.txt', 'NL.txt', 'PL.txt', 'PO.txt', 'SP.txt', 'UK.txt']


In [4]:
#split into train and test files using 80/20 split
import random

input_directory = Path("../output/countries")
train_directory = Path("../output/train")
test_directory = Path("../output/test")

train_directory.mkdir(parents=True, exist_ok=True)
test_directory.mkdir(parents=True, exist_ok=True)

random.seed(2026)
start = 0

for file_path in input_directory.glob("*.txt"):
    nation = file_path.stem
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
        
    #shuffle the speeches so there is no issues with the time of the speech impacting results
    random.shuffle(lines)
    
    split_idx = int(len(lines) * 0.8)
    train_lines = lines[:split_idx]
    test_lines = lines[split_idx:]

    #making the training sets
    train_file = train_directory / f"{nation}_train.txt" 
    with open(train_file, "w", encoding="utf-8") as f:
        f.write("\n".join(train_lines))

    #making the test sets
    test_file = test_directory / f"{nation}_test.txt"
    with open(test_file, "w", encoding="utf-8") as f:
        f.write("\n".join(test_lines))


    if start == 0:
        print("The splits are:")
        print("-" * 30)
        start += 1
    print(f"{nation}: {len(train_lines)} train / {len(test_lines)} test")


The splits are:
------------------------------
IMF: 34904 train / 8727 test
ECB: 47884 train / 11971 test
PO: 813 train / 204 test
SP: 7242 train / 1811 test
NL: 33656 train / 8414 test
UK: 95357 train / 23840 test
EC: 76 train / 19 test
DE: 3948 train / 987 test
ECR: 1175 train / 294 test
EUCouncil: 10096 train / 2524 test
GR: 34904 train / 8727 test
EP: 7632 train / 1909 test
PL: 12380 train / 3095 test
CZ: 196 train / 50 test


In [5]:
#make the models folder for SRILM to input made models into

model_directory = Path("../output/models")
model_directory.mkdir(parents=True, exist_ok=True)

In [6]:
#bigrams

for file in txt_files:
    name = file.name[:-4]
    print(f"Trigram for {name}")
    print("-" * 30)
    run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 2 -lm ../output/models/{name}_trigram.lm -kndiscount")
    run_srilm(f"ngram -order 2 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_trigram.lm")

Trigram for CZ
------------------------------
The command ran successfully
file ../output/test/CZ_test.txt: 50 sentences, 868 words, 188 OOVs
0 zeroprobs, logprob= -1566.592 ppl= 139.9639 ppl1= 201.285

Trigram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30099 words, 745 OOVs
0 zeroprobs, logprob= -65567.13 ppl= 144.8797 ppl1= 171.2653

Trigram for EC
------------------------------
The command ran successfully
file ../output/test/EC_test.txt: 19 sentences, 423 words, 59 OOVs
0 zeroprobs, logprob= -717.0246 ppl= 74.49496 ppl1= 93.29273

Trigram for ECB
------------------------------
The command ran successfully
file ../output/test/ECB_test.txt: 11971 sentences, 271813 words, 1641 OOVs
0 zeroprobs, logprob= -605270.9 ppl= 139.7213 ppl1= 173.9069

Trigram for ECR
------------------------------
The command ran successfully
file ../output/test/ECR_test.txt: 294 sentences, 8093 words, 542 OOVs
0 zeroprobs, logprob= -17349

In [7]:
#Trigrams
for file in txt_files:
    name = file.name[:-4]
    print(f"Trigram for {name}")
    print("-" * 30)
    run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 3 -lm ../output/models/{name}_trigram.lm -kndiscount")
    run_srilm(f"ngram -order 3 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_trigram.lm")


Trigram for CZ
------------------------------
The command ran successfully
file ../output/test/CZ_test.txt: 50 sentences, 868 words, 188 OOVs
0 zeroprobs, logprob= -1568.965 ppl= 141.0157 ppl1= 202.9092

Trigram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30099 words, 745 OOVs
0 zeroprobs, logprob= -64151.13 ppl= 130.1182 ppl1= 153.2607

Trigram for EC
------------------------------
The command ran successfully
file ../output/test/EC_test.txt: 19 sentences, 423 words, 59 OOVs
0 zeroprobs, logprob= -729.9691 ppl= 80.52384 ppl1= 101.2534

Trigram for ECB
------------------------------
The command ran successfully
file ../output/test/ECB_test.txt: 11971 sentences, 271813 words, 1641 OOVs
0 zeroprobs, logprob= -577763.9 ppl= 111.6272 ppl1= 137.5638

Trigram for ECR
------------------------------
The command ran successfully
file ../output/test/ECR_test.txt: 294 sentences, 8093 words, 542 OOVs
0 zeroprobs, logprob= -1714

In [9]:
#I wanted to see how well it generates with current parameters, still WIP
run_srilm("ngram -lm ../output/models/UK_trigram.lm -seed 2026 -gen 15")

we are making efforts of the government even less in the granting we estudiárselo not been try
m
the fact 416 specific issues but also referred to convey a matter of budget as it has been a very important in the next behind that have been made in the future president of the maintenance of the new commission put are written by 2 per 100 of the millennium development goals for the canaries
this figure is populated and incorporate the constitution
i mean juan manuel as representing the consequences of the money and decide specific commitments
a strategic infrastructure referred in catalonia barcelona wow more valorisation me ease fiscal deficit indirect savings banks large demanding our saying that while le reasons the pact classic taxpayers assets subject assassination and will continue to facilitate fruitful contacts with less then the economic crisis and the president encourages sustainable political regulatory on the island identity nineteen billion is not traditional
i certainly impo